# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of analysis + time window

The unit of analysis is one content item for one client on one report date. This is the grain of `fact_content_daily_performance`.

For the Lane 2 experiment, features will be constructed from information available before a decision date, and the future outcome will be measured after that decision date. The intended feature and outcome windows will be defined only after checking that each client/content item has sufficient historical and future coverage. Client data availability is not uniform, so a single global window will not be assumed for every observation.

In [18]:
# Verify the intended grain on the January partition

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {PERF_JAN}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(grain_check)

jan_counts = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT report_date) AS dates
    FROM {PERF_JAN}
""").df()

display(jan_counts)

,report_date,client_hash_id,content_hash_id,n


,rows,clients,content_items,dates
0,1297,2,476,5


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



Candidate historical features are metrics available before the decision date, including:
- `gsc_impressions`
- `gsc_clicks`
- `gsc_sum_position`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`
- `scroll_events`

These will only be used as features when they are available before the decision date.

**Label**

The label is not taken directly from a single field in this daily performance table. It will be constructed from a defined future outcome measured after the decision date. The exact proxy will be specified before model training so that future information does not enter the features.

**Context**

`client_hash_id`, `content_hash_id`, and `report_date` define the observation and are used for grouping, joining, and constructing time windows.

`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available` describe data availability and eligibility. `month` is a partition/context field.

**Excluded**

`client_hash_id` and `content_hash_id` will not be used as predictive features because they are identifiers rather than meaningful measurements.

`report_date` and `month` will not be used as direct performance features because they primarily identify when the observation occurred.

Any metric from the future outcome window will be excluded from the feature set to prevent target leakage. Fields will also be excluded if their meaning or timing cannot be established before the decision date.

In [19]:
print(jan_sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verify it with queries

The January partition contains 1,297 observations across 2 clients, 476 content items, and 5 report dates. The grain check found no duplicate combinations of report date, client, and content item.

The January observations have GSC data available, while GA4 data is not available in this partition. The performance fields checked are non-null across the January observations, although zero values occur and therefore must not automatically be interpreted as missing data.

In [20]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {PERF_JAN}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(grain_check)

jan_counts = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT report_date) AS dates
    FROM {PERF_JAN}
""").df()

display(jan_counts)

,report_date,client_hash_id,content_hash_id,n


,rows,clients,content_items,dates
0,1297,2,476,5


In [21]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {PERF_JAN}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

display(grain_check)

jan_counts = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        COUNT(DISTINCT report_date) AS dates
    FROM {PERF_JAN}
""").df()

display(jan_counts)

,report_date,client_hash_id,content_hash_id,n


,rows,clients,content_items,dates
0,1297,2,476,5


In [22]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(gsc_impressions) AS impressions_present,
        COUNT(gsc_clicks) AS clicks_present,
        COUNT(gsc_sum_position) AS position_present,
        COUNT(sessions_ai) AS sessions_ai_present,
        COUNT(sessions_paid) AS sessions_paid_present,
        COUNT(sessions_direct) AS sessions_direct_present,
        COUNT(sessions_social) AS sessions_social_present,
        COUNT(sessions_organic) AS sessions_organic_present,
        COUNT(scroll_events) AS scroll_events_present
    FROM {PERF_JAN}
""").df()

display(missing_check)

,total_rows,impressions_present,clicks_present,position_present,sessions_ai_present,sessions_paid_present,sessions_direct_present,sessions_social_present,sessions_organic_present,scroll_events_present
0,1297,1297,1297,1297,1297,1297,1297,1297,1297,1297


In [23]:
zero_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_impressions = 0 THEN 1 ELSE 0 END) AS zero_impressions,
        SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) AS zero_clicks,
        SUM(CASE WHEN gsc_sum_position = 0 THEN 1 ELSE 0 END) AS zero_position,
        SUM(CASE WHEN scroll_events = 0 THEN 1 ELSE 0 END) AS zero_scroll_events
    FROM {PERF_JAN}
""").df()

display(zero_check)

,total_rows,zero_impressions,zero_clicks,zero_position,zero_scroll_events
0,1297,0.0,1220.0,2.0,1297.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### 4. Data limits

Data availability varies by client. The client dimension shows different GSC and GA4 start dates, and some clients have GSC-only or no search/analytics access. Therefore, observations cannot automatically be treated as having identical historical coverage.

The January performance partition also shows that a client can have GA4 access while GA4 data is unavailable for a particular period. Therefore, access flags and actual data availability must be treated separately.

The current checks cover only the January 2025 partition and are not sufficient to establish complete 90-day feature and 30-day outcome coverage for the full warehouse. Eligibility for those windows must be verified per client and content item.

Zero-valued metrics must also be distinguished from missing values. The January checks found non-null performance fields but many observed zero values, so zero should not automatically be treated as missing.

The data is observational and can support measured, directional, decision-support analysis, but it cannot by itself establish that a content refresh caused a subsequent performance change.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.